In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from source_code.itransformer_dataset import load_and_preprocess_data
import pandas as pd

In [ ]:
# --- Step A: Load Data ---
df_long, hist_exog_cols, TARGETS = load_and_preprocess_data("/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_planck_weather_ts.csv")

In [ ]:
df_long.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import iTransformer
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse

# --- Step B: Train-Test Split (Long Format) ---
HORIZON = 144  # Prediksi 24 timestep ke depan (4 jam)
INPUT_SIZE = 288  # 96 timestep ke belakang (16 jam)
NUM_TARGETS = len(TARGETS)
# Ambil 10.000 titik waktu terakhir, untuk menghemat sumber daya, karena jika data dipakai semua maka itu akan banyak sekali
NUM_TIMESTAMPS = 10000  
TEST_TIMESTAMPS = 2000  # 2.000 titik waktu untuk pengujian

# Ambil 140.000 baris terakhir (10.000 timestamp x 14 fitur)
df_sample = df_long.tail(NUM_TIMESTAMPS * NUM_TARGETS).reset_index(drop=True)

# Cari titik batas pemisah (split) berdasarkan timestamp ds
unique_timestamps = df_sample['ds'].unique()
last_train_ds = unique_timestamps[-TEST_TIMESTAMPS] # tanggal 2000 pertama

train_df = df_sample[df_sample['ds'] < last_train_ds].copy()
test_df = df_sample[df_sample['ds'] >= last_train_ds].copy()

print(
    f'\nInformasi Dataset:\n'
    f'- Jumlah Fitur Target : {NUM_TARGETS} fitur\n'
    f'- Timestep Train Data  : {len(train_df) // NUM_TARGETS} baris per fitur\n'
    f'- Timestep Test Data   : {len(test_df) // NUM_TARGETS} baris per fitur'
)

# --- Step C: Inisialisasi Model iTransformer ---
print('\n2. Menginisialisasi Model iTransformer...')

models = [
    iTransformer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=NUM_TARGETS,
        max_steps=2000,  # Ubah ke 1000+ jika ingin hasil optimal
        batch_size=32,
        learning_rate=0.001,
        # -----------------------------------------------------------------
        # OPTIMASI VALIDASI:
        val_check_steps=50,  # 👈 Validasi hanya dilakukan setiap 50 steps
        early_stop_patience_steps=3,  # Toleransi 3x pengecekan validasi (3 x 50 = 150 steps)
        # -----------------------------------------------------------------
        scaler_type='standard',  # Normalisasi data otomatis per unique_id
        accelerator ='auto',  # Otomatis deteksi GPU/CPU
        hist_exog_list=hist_exog_cols,
    )
]

# Inisialisasi runner NeuralForecast
nf = NeuralForecast(models=models, freq='10m')

# --- Step D: Fit Model ---
VAL_SIZE = 144

print('\n3. Memulai proses Training (Fit)...')
nf.fit(df=train_df, val_size = VAL_SIZE)

# --- Step D.1: Menyimpan Model yang Sudah Dilatih ---
print('\n3.b. Menyimpan model iTransformer...')

# Tentukan folder lokasi penyimpanan
MODEL_PATH = '/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_plank_weather/all_models_training'

# Simpan model (overwrite=True agar menimpa model lama jika ada)
nf.save(path=MODEL_PATH, overwrite=True)

print(f'✅ Model berhasil disimpan di folder: {MODEL_PATH}')

# --- Step E: Cross-Validation / Predict ---
print('\n4. Melakukan Prediksi pada Data Uji (14 Fitur)...')

# Hitung jumlah window prediksi (2000 / 24 = 83 windows)
N_WINDOWS = TEST_TIMESTAMPS // HORIZON

cv_df = nf.cross_validation(
    df=df_sample,
    n_windows=N_WINDOWS,
    test_size=None,  # 👈 Kosongkan test_size secara eksplisit
    val_size=VAL_SIZE,
    use_fitted=True,  # Gunakan model yang sudah di-fit pada Step D
)

print('\nHasil Prediksi (5 baris pertama):')
print(cv_df.head())

# --- Step F: Evaluasi Metrik (MAE & RMSE) ---
print('\n5. Menghitung Metrik Evaluasi per Fitur...')

# 1. Jalankan evaluasi per fitur seperti biasa
evaluation_results = evaluate(
    df=cv_df, metrics=[mae, rmse], models=['iTransformer']
)

# 2. Pivot agar 1 fitur = 1 baris, dengan kolom MAE dan RMSE di sampingnya
feature_scores = (
    evaluation_results.groupby(['unique_id', 'metric'])['iTransformer']
    .mean()
    .unstack('metric')
    .reset_index()
)

feature_scores.columns.name = None
print(feature_scores)

print('=== SKOR PREDIKSI PER FITUR (SUHU) ===')
# Urutkan dari fitur yang MAE-nya paling tinggi (paling sulit diprediksi)
feature_scores_sorted = feature_scores.sort_values(
    by='mae', ascending=False
)
print(feature_scores_sorted)

# --- Step G: Visualisasi Hasil Prediksi (Grid 7x2) ---
print('\n6. Menampilkan Grafik Hasil Prediksi T (degC)')

# Ambil 200 titik waktu terakhir per fitur untuk visualisasi
plot_df = cv_df.groupby('unique_id').tail(200)

fig, axes = plt.subplots(figsize=(16, 10))

for idx, target_name in enumerate(TARGETS):
  sub_df = plot_df[plot_df['unique_id'] == target_name]
  ax = axes

  ax.plot(
      sub_df['ds'], sub_df['y'], label='Nilai Asli', color='black', alpha=0.8
  )
  ax.plot(
      sub_df['ds'],
      sub_df['iTransformer'],
      label='Prediksi iTransformer',
      color='red',
      linestyle='--',
  )

  ax.set_title(f'Fitur: {target_name}', fontsize=10, fontweight='bold')
  ax.grid(True, linestyle=':', alpha=0.6)
  if idx == 0:
    ax.legend()

plt.suptitle(
    'iTransformer: Hasil Prediksi T (degC) (Max Planck)',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('prediksi_1_fitur_itransformer_2.png')
print(
    'Grafik berhasil disimpan'
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from neuralforecast import NeuralForecast
# Import 5 model yang akan dibandingkan
from neuralforecast.models import GRU, DLinear, NHiTS, PatchTST, iTransformer
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse

# --- Step B: Train-Test Split (Long Format) ---
HORIZON = 144  # Prediksi 144 jam ke depan
INPUT_SIZE = 288  # Lookback 288 jam ke belakang
NUM_TARGETS = len(TARGETS)

# Ambil sample timestamp
NUM_TIMESTAMPS = 10000
TEST_TIMESTAMPS = 2000

df_sample = df_long.tail(NUM_TIMESTAMPS * NUM_TARGETS).reset_index(drop=True)

# Cari titik pemisah (split) berdasarkan ds
unique_timestamps = df_sample['ds'].unique()
last_train_ds = unique_timestamps[-TEST_TIMESTAMPS]

train_df = df_sample[df_sample['ds'] < last_train_ds].copy()
test_df = df_sample[df_sample['ds'] >= last_train_ds].copy()

print(
    f'\nInformasi Dataset:\n'
    f'- Jumlah Fitur Target : {NUM_TARGETS} fitur\n'
    f'- Timestep Train Data  : {len(train_df) // NUM_TARGETS} baris per fitur\n'
    f'- Timestep Test Data   : {len(test_df) // NUM_TARGETS} baris per fitur'
)

# --- Step C: Inisialisasi 5 Model (Multi-Model Training) ---
print('\n2. Menginisialisasi 5 Model (iTransformer, PatchTST, DLinear, N-HiTS, GRU)...')

# Nama-nama model untuk evaluasi otomatis nanti
MODEL_NAMES = ['iTransformer', 'PatchTST', 'DLinear', 'NHiTS', 'GRU']

models = [
    # 1. iTransformer (Transformer-based, tanpa hist_exog_list)
    iTransformer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=NUM_TARGETS,
        max_steps=2000,
        batch_size=32,
        learning_rate=0.001,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    # 2. PatchTST (Patching Transformer, tanpa hist_exog_list)
    PatchTST(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=NUM_TARGETS,
        max_steps=2000,
        batch_size=32,
        learning_rate=0.001,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    # 3. DLinear (Mendukung hist_exog_list)
    DLinear(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols,
        max_steps=2000,
        batch_size=32,
        learning_rate=0.001,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    # 4. N-HiTS (Mendukung hist_exog_list)
    NHiTS(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols,
        max_steps=2000,
        batch_size=32,
        learning_rate=0.001,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
    # 5. GRU (Recurrent Neural Network, Mendukung hist_exog_list)
    GRU(
        h=HORIZON,
        input_size=INPUT_SIZE,
        hist_exog_list=hist_exog_cols,
        max_steps=2000,
        batch_size=32,
        learning_rate=0.001,
        val_check_steps=50,
        early_stop_patience_steps=3,
        scaler_type='standard',
        accelerator='auto',
    ),
]

# Inisialisasi runner NeuralForecast dengan frekuensi per jam ('h')
nf = NeuralForecast(models=models, freq='h')

# --- Step D: Fit All Models ---
VAL_SIZE = 144

print('\n3. Memulai proses Training 5 Model sekaligus...')
nf.fit(df=train_df, val_size=VAL_SIZE)

# --- Step D.1: Menyimpan Seluruh Model ---
print('\n3.b. Menyimpan model-model yang telah dilatih...')
MODEL_PATH = '/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_plank_weather/all_models_training'
nf.save(path=MODEL_PATH, overwrite=True)
print(f'✅ Semua model berhasil disimpan di: {MODEL_PATH}')

# --- Step E: Cross-Validation / Predict ---
print('\n4. Melakukan Prediksi untuk Semua Model...')
N_WINDOWS = TEST_TIMESTAMPS // HORIZON

cv_df = nf.cross_validation(
    df=df_sample,
    n_windows=N_WINDOWS,
    test_size=None,
    val_size=VAL_SIZE,
    use_fitted=True,
)

print('\nHasil Prediksi 5 Model (5 baris pertama):')
print(cv_df.head())

# --- Step F: Evaluasi Metrik Komparasi (MAE & RMSE) ---
print('\n5. Menghitung & Membandingkan Metrik Evaluasi 5 Model...')

# Evaluasi 5 model sekaligus
evaluation_results = evaluate(
    df=cv_df, metrics=[mae, rmse], models=MODEL_NAMES
)

# Merapikan tabel komparasi skor antar model
score_summary = (
    evaluation_results.groupby(['unique_id', 'metric'])[MODEL_NAMES]
    .mean()
    .unstack('metric')
)

print('=== PERBANDINGAN PERFORMA MODEL (MAE & RMSE) ===')
print(score_summary)

# --- Step G: Visualisasi Perbandingan Hasil Prediksi ---
print('\n6. Menampilkan Grafik Komparasi Hasil Prediksi T (degC)...')

# Ambil 200 titik waktu terakhir untuk plot agar mudah dibaca
plot_df = cv_df.groupby('unique_id').tail(200)
sub_df = plot_df[plot_df['unique_id'] == TARGETS[0]]  # Visualisasi Suhu T (degC)

plt.figure(figsize=(16, 7))

# Visualisasi Nilai Asli
plt.plot(
    sub_df['ds'],
    sub_df['y'],
    label='Nilai Asli (Actual)',
    color='black',
    linewidth=2,
    alpha=0.8,
)

# Warna khas untuk masing-masing model
colors = {
    'iTransformer': 'red',
    'PatchTST': 'blue',
    'DLinear': 'green',
    'NHiTS': 'orange',
    'GRU': 'purple',
}

# Plot garis prediksi untuk setiap model
for model_name in MODEL_NAMES:
    if model_name in sub_df.columns:
        plt.plot(
            sub_df['ds'],
            sub_df[model_name],
            label=f'Prediksi {model_name}',
            color=colors.get(model_name, 'gray'),
            linestyle='--',
            alpha=0.7,
        )

plt.title(
    'Perbandingan Prediksi Suhu T (degC): iTransformer vs PatchTST vs DLinear vs N-HiTS vs GRU',
    fontsize=12,
    fontweight='bold',
)
plt.xlabel('Waktu (Timestamp)', fontsize=10)
plt.ylabel('Suhu T (degC)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left')
plt.tight_layout()

plt.savefig('komparasi_5_model_prediksi_suhu.png')
print('✅ Grafik perbandingan 5 model berhasil disimpan sebagai komparasi_5_model_prediksi_suhu.png')
plt.show()